In [1]:
import json
import urllib.request
import time

In [2]:
INPUT_FILE = "wyniki.json"
OUTPUT_FILE = "wyniki_with_cwe.json"

In [23]:
def load_stacked_json(filepath):
    all_data = []
    with open(filepath, 'r') as file:
        content = file.read()

    decoder = json.JSONDecoder()
    idx = 0
    length = len(content)

    while idx < length:
        while idx < length and content[idx].isspace():
            idx += 1
        if idx >= length:
            break

        array_data, idx = decoder.raw_decode(content, idx)
        all_data.extend(array_data)

    return all_data

In [24]:
data = load_stacked_json(INPUT_FILE)

In [30]:
for i, alert in enumerate(data):
    cve_id = alert.get("Vulnerability", "")
    if cve_id.startswith("CVE-") and "CWE" not in alert:
        alert["CWE"] = fetch_cwe_from_circl(cve_id)
    else:
        alert["CWE"] = "CWE-UNKNOWN"
    print(alert["CWE"])

    if i % 500 == 0 and i > 0:
        print(f"Processed {i} / {len(data)}...")
        time.sleep(1) # Be polite to the API

with open(OUTPUT_FILE, 'w') as f:
    json.dump(data, f, indent=4)
print(f"Done! Saved to {OUTPUT_FILE}")


KeyboardInterrupt: 